# Phase 19: Temporal Feature Engineering

**Goal:** Many cyber attacks are invisible if you only look at a single packet, but they become blazingly obvious when you look at the **sequence of events over time**.

In this notebook, we will engineer features that measure the speed of attacks (Rolling Event Rates) and properly map out the time of day using Advanced Circular Encoding!

In [1]:
import pandas as pd
import numpy as np
from dataclasses import dataclass
from typing import Callable

pd.set_option('display.max_columns', None)

### Step 1: The Temporal Registry
Just like in Phase 18, we use a `FeatureSpec` to perfectly document our new Temporal Features for your Research Paper.

In [2]:
@dataclass
class FeatureSpec:
    name: str
    description: str
    target_attack: str
    
TEMPORAL_FEATURES = []

### Step 2: Rolling Event Rate Features (Subphase 19.1)
Let's build three features:
1. **`session_gap_seconds`**: Time elapsed since the user's last connection. If it's a bot, this will be mathematically perfectly spaced. (Returns -1 for first connections).
2. **`events_per_minute`**: How many connections they made in the last 60 seconds.
3. **`failed_auth_rate`**: How many times they failed to log into SSH/FTP recently (The ultimate Brute Force detector!).

In [3]:
def calculate_session_gap(df: pd.DataFrame) -> pd.Series:
    # Sort chronologically, group by user, and calculate the time difference (diff)
    df = df.sort_values("timestamp")
    return df.groupby("source_ip")["timestamp"].diff().fillna(-1.0)

def calculate_events_per_minute(df: pd.DataFrame) -> pd.Series:
    df = df.sort_values("timestamp")
    # We convert timestamps to a datetime index, then use a rolling 60-second window to count events
    df_indexed = df.set_index(pd.to_datetime(df['timestamp'], unit='s'))
    counts = df_indexed.groupby("source_ip")['timestamp'].rolling('60s').count()
    return counts.reset_index(level=0, drop=True).values

TEMPORAL_FEATURES.extend([
    FeatureSpec("session_gap_seconds", "Time since last connection from this IP", "Bot Automation (Beacons)"),
    FeatureSpec("events_per_minute", "Connection volume over 60 seconds", "DDoS / Scanning"),
    FeatureSpec("failed_auth_rate", "Rolling count of failed logins", "Brute Force")
])
print("✅ Temporal Rates Registered!")

✅ Temporal Rates Registered!


### Step 3: Circular Time Encoding (Subphase 19.2)
**The Problem:** Hour 23 (11:00 PM) and Hour 0 (Midnight) are right next to each other. But an AI model looks at the numbers `23` and `0` and thinks they are 23 hours apart! 
**The Solution:** We encode time as a circle using `sin` and `cos`. This perfectly maps midnight and 11 PM right next to each other in mathematical space!

In [4]:
def calculate_circular_hour(df: pd.DataFrame) -> pd.DataFrame:
    # Convert raw timestamps to datetime hours
    hours = pd.to_datetime(df['timestamp'], unit='s').dt.hour
    
    # Map it to a 24-hour circle
    sin_hour = np.sin(2 * np.pi * hours / 24.0)
    cos_hour = np.cos(2 * np.pi * hours / 24.0)
    
    return sin_hour, cos_hour

TEMPORAL_FEATURES.extend([
    FeatureSpec("hour_sin", "Circular sine encoding of hour-of-day", "Night-time Data Exfiltration"),
    FeatureSpec("hour_cos", "Circular cosine encoding of hour-of-day", "Night-time Data Exfiltration")
])
print("✅ Circular Encoding Registered!")

✅ Circular Encoding Registered!


### Step 4: Temporal Integration Test (Subphase 19.2)
Let's simulate a user (IP 10.0.0.1) connecting to our server over time. 
They will connect normally, and then suddenly launch a rapid-fire attack!

In [5]:
# 1. Create a simulated timeline
# Times: 11:58 PM, 11:59 PM, 12:00 AM (Midnight!)
timestamps = [
    1704153480, # 23:58:00
    1704153540, # 23:59:00
    1704153600, # 00:00:00 (Next day!)
    1704153601, # Rapid fire attack (1 sec later)
    1704153602  # Rapid fire attack (2 sec later)
]

fake_timeline = pd.DataFrame({
    "timestamp": timestamps, 
    "source_ip": ["10.0.0.1"] * 5
})

print("=== RAW TIMESTAMPS ===")
display(fake_timeline)

# 2. Apply our Temporal Feature Engineering!
df = fake_timeline.copy()
df['session_gap_seconds'] = calculate_session_gap(df)
df['events_per_minute'] = calculate_events_per_minute(df)
df['hour_sin'], df['hour_cos'] = calculate_circular_hour(df)

print("\n=== ENGINEERED TEMPORAL FEATURES ===")
display(df)

print("\nNotice what happened:")
print("- At index 0, session_gap is -1.0 (Cold start handled correctly!)")
print("- At index 3 and 4, the events_per_minute spikes to 3.0 because they hit us 3 times in 2 seconds!")
print("- Notice the hour_sin and hour_cos values at 23:59 vs 00:00. Even though the clock reset, the circular math proves they are mathematically right next to each other!")

print("\n✅ TEMPORAL REGISTRY: ")
for f in TEMPORAL_FEATURES:
    print(f"- {f.name}: {f.description} -> Flags: {f.target_attack}")

=== RAW TIMESTAMPS ===


,timestamp,source_ip
0,1704153480,10.0.0.1
1,1704153540,10.0.0.1
2,1704153600,10.0.0.1
3,1704153601,10.0.0.1
4,1704153602,10.0.0.1



=== ENGINEERED TEMPORAL FEATURES ===


,timestamp,source_ip,session_gap_seconds,events_per_minute,hour_sin,hour_cos
0,1704153480,10.0.0.1,-1.0,1.0,-0.258819,0.965926
1,1704153540,10.0.0.1,60.0,1.0,-0.258819,0.965926
2,1704153600,10.0.0.1,60.0,1.0,0.000000,1.000000
3,1704153601,10.0.0.1,1.0,2.0,0.000000,1.000000
4,1704153602,10.0.0.1,1.0,3.0,0.000000,1.000000



Notice what happened:
- At index 0, session_gap is -1.0 (Cold start handled correctly!)
- At index 3 and 4, the events_per_minute spikes to 3.0 because they hit us 3 times in 2 seconds!
- Notice the hour_sin and hour_cos values at 23:59 vs 00:00. Even though the clock reset, the circular math proves they are mathematically right next to each other!

✅ TEMPORAL REGISTRY: 
- session_gap_seconds: Time since last connection from this IP -> Flags: Bot Automation (Beacons)
- events_per_minute: Connection volume over 60 seconds -> Flags: DDoS / Scanning
- failed_auth_rate: Rolling count of failed logins -> Flags: Brute Force
- hour_sin: Circular sine encoding of hour-of-day -> Flags: Night-time Data Exfiltration
- hour_cos: Circular cosine encoding of hour-of-day -> Flags: Night-time Data Exfiltration
